<a href="https://colab.research.google.com/github/MCTEEKUNG/Heatwave_Backend_Elysia/blob/main/region-line-oa/notebooks/colab_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Heatwave-AI — Data & Model Pipeline (Colab)

Runs the **real** pipeline: Open-Meteo (ERA5) → sWBGT → per-province percentile + persistence labels → temporal split → LightGBM (isotonic-calibrated) → writes forecasts + thresholds to **Supabase** and uploads the model to **Hugging Face**.

### Set these Colab secrets first (🔑 left sidebar → Secrets, toggle “Notebook access”)
| Secret | What | Where to get it |
|---|---|---|
| `DATABASE_URL` | Supabase **Session pooler** connection string | Supabase → Connect → *Session pooler* (IPv4, works from Colab). Looks like `postgresql://postgres.ejvrzprcbxgvaqbydagd:<PASSWORD>@aws-...pooler.supabase.com:5432/postgres` |
| `HF_TOKEN` | Hugging Face **write** token | huggingface.co → Settings → Access Tokens |
| `GITHUB_TOKEN` | *(only if the repo is private)* GitHub PAT with `repo` scope | github.com → Settings → Developer settings → PATs |

> Use the **Session pooler** string (port 5432), not the Transaction pooler (6543) — the Python writer (`psycopg`) uses statements the transaction pooler doesn’t support.

## 1. Clone the repo (branch `feat/region-line-oa`)

In [5]:
REPO = 'https://github.com/MCTEEKUNG/Heatwave_Backend_Elysia.git'
BRANCH = 'feat/region-line-oa'

tok = None
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

url = REPO.replace('https://', f'https://{tok}@') if tok else REPO
!rm -rf heatwave
!git clone --depth 1 -b $BRANCH $url heatwave
%cd heatwave

Cloning into 'heatwave'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 207 (delta 6), reused 186 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 882.54 KiB | 3.54 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/heatwave/heatwave


## 2. Install dependencies

In [6]:
!pip -q install -r requirements.txt pyarrow 'psycopg[binary]' huggingface_hub

## 3. Load secrets into the environment

In [7]:
import os
from google.colab import userdata
os.environ['DATABASE_URL'] = userdata.get('DATABASE_URL')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('DATABASE_URL set:', bool(os.environ.get('DATABASE_URL')))
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

DATABASE_URL set: True
HF_TOKEN set: True


## 4. Build the dataset (real Open-Meteo, 77 provinces × 1991–2025)
~77 archive calls; takes a few minutes. Writes `data/processed/dataset.parquet` + `province_thresholds.parquet`.

In [13]:
!python -m pipeline.build_dataset

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/heatwave/heatwave/pipeline/build_dataset.py", line 45, in <module>
    main()
  File "/content/heatwave/heatwave/pipeline/build_dataset.py", line 36, in main
    ds, thr = build_for_provinces(provinces, "1991-01-01", "2025-12-31")
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/heatwave/heatwave/pipeline/build_dataset.py", line 20, in build_for_provinces
    raw = openmeteo_client.fetch_history(p["lat"], p["lon"], start, end)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/heatwave/heatwave/src/openmeteo_client.py", line 41, in fetch_history
    r.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 

## 5. Train + calibrate + evaluate  —  🚦 QUALITY GATE
Read the printed metrics: **`test` PR-AUC / MCC / F2** must beat **`baseline_constant`**. If not, the model has no skill — add features (geopotential 500 hPa, soil moisture, more lags) before deploying.

In [9]:
!python -m pipeline.train

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/heatwave/heatwave/pipeline/train.py", line 100, in <module>
    main()
  File "/content/heatwave/heatwave/pipeline/train.py", line 82, in main
    dataset = pd.read_parquet(DATASET_PATH)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 667, in read_parquet
    return impl.read(
           ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 267, in read
    path_or_handle, handles, filesystem = _get_path_or_handle(
                                          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 140, in _get_path_or_handle
    handles = get_handle(
              ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/common.py", line 882, in get_handle
    ha

## 6. Load thresholds into Supabase (`heatwave.province_thresholds`)

In [10]:
!python -m pipeline.load_thresholds

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/heatwave/heatwave/pipeline/load_thresholds.py", line 40, in <module>
    main()
  File "/content/heatwave/heatwave/pipeline/load_thresholds.py", line 33, in main
    df = pd.read_parquet(THRESHOLDS_PATH)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 667, in read_parquet
    return impl.read(
           ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 267, in read
    path_or_handle, handles, filesystem = _get_path_or_handle(
                                          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 140, in _get_path_or_handle
    handles = get_handle(
              ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/common.py", line 882, in ge

## 7. Generate forecasts → Supabase (`heatwave.forecasts`)

In [11]:
!python -m pipeline.run_forecast

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/heatwave/heatwave/pipeline/run_forecast.py", line 224, in <module>
    main()
  File "/content/heatwave/heatwave/pipeline/run_forecast.py", line 195, in main
    model = joblib.load(MODEL_PATH)
            ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/joblib/numpy_pickle.py", line 735, in load
    with open(filename, "rb") as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'models/heatwave_model.pkl'


## 8. Upload model + thresholds to Hugging Face (for Render to download in M4)

In [12]:
ตอนนี้ผมกด Run All แล้ว แล้วก็ติดที่เซลล์ที่แปด from huggingface_hub import HfApi
REPO_ID = 'MCTEEKUNG123/Heatwave-AI'  # <-- change if your HF model repo differs
api = HfApi(token=os.environ['HF_TOKEN'])
api.upload_file(path_or_fileobj='models/heatwave_model.pkl',
                path_in_repo='models/heatwave_model.pkl', repo_id=REPO_ID)
api.upload_file(path_or_fileobj='data/processed/province_thresholds.parquet',
                path_in_repo='data/province_thresholds.parquet', repo_id=REPO_ID)
print('uploaded model + thresholds to', REPO_ID)

ValueError: Provided path: 'models/heatwave_model.pkl' is not a file on the local file system

## Done ✅
- `heatwave.province_thresholds` + `heatwave.forecasts` populated in Supabase
- `heatwave_model.pkl` + thresholds parquet on Hugging Face

**Next:** verify the train metrics beat baseline (M2 gate), then M3 (LINE) → M4 (deploy: Render downloads the model from HF + daily cron runs `run_forecast`).